### Simulation Study: Within-Patient vs. Between-Patient Variability in Cross-over Trials

From Hills & Armitage (1979), we can see that analyzing a cross-over trial via group differences nicely eliminates between-patient variability, while analyzing via overall difference means this variability needs to be included (increasing standard deviation on error and decreasing precision).

Brief:
Build a simulation study that implements the Hills & Armitage (1979) proof in code. 
1. Generate simulated data
2. Implement both analysis strategies (calculating group differences vs. overall difference)
3. See if we can replicate the analytical result from the paper in code.

#### Setup

In [1]:
# Importing libraries
import numpy as np
import pandas as pd
import scipy.stats as stats

# Random seed for reproducibility
seed = np.random.seed(929)

#### Trial Parameters

Let: 

$e =$ response in the absence of treatment (i.e., baseline)

Then:

$e_1 = $ Baseline response in Period 1

$e_2 = $ Baseline response in Period 2

and

$T_x, T_y = $ Fixed amounts by which treatments change response

Then the trial outcome can be modeled:

Group A (X,Y)

$y_1 = e_1 + T_x$

$y_2 = e_2 + T_y$

Group B (Y,X)

$y_1 = e_1 + T_y$

$y_2 = e_2 + T_x$

Note: Values of $y_1, y_2, e_1, e_2$ change for each subject, but values of $T_x, T_y$ are constant.


In [ ]:
# Using Example 1 from Hills & Armitage (1979) for loose inspiration

# Sample sizes
n_A = 17        # Group A (X, Y)
n_B = 12        # Group B (Y, X)

# True fixed treatment effects (defining these as known constants so we have a reference to compare our simulations to, in real life we wouldn't necessarily know these)
T_x = 5.0                   # fixed treatment effect for Treatment X
T_y = 2.0                   # fixed treatment effect for Treatment Y

# Defining the true relative treatment effect as the difference between the two true parameters
true_treatment_effect = T_x - T_y

# Other parameters
# Again, just defining these as known up-front so we can compare our simulation results to actual values we know. In real life we would have to figure these out from the data, but here, to simulate data, we need some parameters to simulate from.

# Let (e_1 - e_2) be the time trend for a patient
# If there is any consistent time trend, the values of (e_1 - e_2) over all patients will have a non-zero mean of delta w/ standard deviation sigma
delta = 1.5                 # non-zero mean of the time trends
sigma_within = 3.0          # stdev of individual time trends around mean delta (within-patient variability)
                            # epsilon (each individual patient's random noise) gets folded in here (Hills & Armitage don't have this term, but it's gotta go somewhere)

mu_baseline = 50.0          # overall population mean of all the time trends (e_1 - e_2), or avg baseline change 
                            # between periods 1 and 2

sigma_between = 10.0        # variation of mu_baseline or how much patients' baselines differ across the population (between-patient variability


#### Simulating Trial Data

Patients are randomly allocated to Group A or Group B and assessed during each period. 

- Group A gets Treatment X in Period 1, then Treatment Y in Period 2.
- Group B gets Treatment Y in Period 1, then Treatment X in Period 2. 

In [3]:
def simulate_crossover_data(n_A, n_B, T_x, T_y, delta, sigma_within, sigma_between, mu_baseline):
    """
    A function that generates simulated cross-over trial data based on Hills & Armitage (1979).
    """

    # Simulating patient-specific biological baselines via random draws from a normal distribution with mean mu_baseline and stdev sigma_between (since diff patients)
    baselines_A = np.random.normal(mu_baseline, sigma_between, n_A)
    baselines_B = np.random.normal(mu_baseline, sigma_between, n_B)

    # Simulating time trend (e_1 - e_2), which should have a mean of delta and a stdev of sigma_within (variation within patients)

    # First we start by generating random noise epsilon via random draws from a normal distribution with mean 0 and stdev ... 

    # QUESTION why divide sigma_within by sqrt(2)?
    # "We apply delta entirely to Period 1 for mathematical simplicity, and split variance"

    # Group A
    eps_1A = np.random.normal(0, sigma_within / np.sqrt(2), n_A)
    eps_2A = np.random.normal(0, sigma_within / np.sqrt(2), n_A)

    # Group B
    eps_1B = np.random.normal(0, sigma_within / np.sqrt(2), n_B)
    eps_2B = np.random.normal(0, sigma_within / np.sqrt(2), n_B)

    # Now calculating e values as a sum of baseline, delta, and epsilon
    # QUESTION: Why restrict delta just to the 1 baselines, why not also include in the 2 terms?

    # Group A
    e1_A = baselines_A + delta + eps_1A
    e2_A = baselines_A + eps_2A

    # Group B
    e1_B = baselines_B + delta + eps_1B
    e2_B = baselines_B + eps_2B

    # Calculating patient responses y1 and y2

    # Group A (X,Y)
    y1_A = e1_A + T_x
    y2_A = e2_A + T_y

    # Group B (Y,X)
    y1_B = e1_B + T_y
    y2_B = e2_B + T_x

    # Packaging simulated data into nice dataframes
    df_A = pd.DataFrame({
        'Group': 'A', 
        'y1': y1_A, 
        'y2': y2_A
        })
    
    df_B = pd.DataFrame({
        'Group': 'B',
        'y1': y1_B,
        'y2': y2_B
    })

    # Returning the concatenated dataframes
    return pd.concat(
        [df_A, df_B], 
        ignore_index=True       # clean index in new df
        )

# Calling the function to simulate data for a crossover trial with the given parameters
df_trial = simulate_crossover_data(
    n_A,
    n_B,
    T_x,
    T_y,
    delta, 
    sigma_within,
    sigma_between,
    mu_baseline
)

# Inspecting the simulated data, 10 random points each from Group A and Group B
df_trial.groupby('Group').sample(n=10, random_state=seed)

,Group,y1,y2
3,A,54.953197,52.338053
8,A,28.190986,24.102469
0,A,46.404976,42.221202
2,A,62.602314,55.592975
7,A,45.036365,43.083907
10,A,48.857166,51.607163
15,A,70.519386,69.371499
9,A,53.488516,47.883723
16,A,57.526934,54.900382
12,A,43.134815,40.068534


#### Variance Helper Functions

In [4]:
def calculate_standard_error(var1, var2, n1, n2, method, dof=None, dof_penalty_1=1, dof_penalty_2=1, is_group_diff=False):

    """
    Helper function to calculate standard error via both pooled and unpooled methods.

    """ 

    # Assuming the stdevs estimate the same thing, we can pool them
    if method == 'pooled':

        # Calculating total penalty across entire model
        total_penalty = dof_penalty_1 + dof_penalty_2

        # Calculating pooled variance weighted by dofs
        s2_pooled = ((n1 - dof_penalty_1) * var1 + (n2 - dof_penalty_2) * var2) / (n1 + n2 - total_penalty)        
        var_term = (s2_pooled / n1) + (s2_pooled / n2)
        auto_dof = n1 + n2 - total_penalty

    # QUESTION: what is welch-satterthwaite and is it the same as normal deviate / the hills-armitage approach?
    elif method == 'unpooled':
        var_term = (var1 / n1) + (var2 / n2)

        # Calculating separate variances
        df_num = var_term**2
        df_den = ((var1 / n1)**2 / (n1 - dof_penalty_1)) + ((var2 / n2)**2 / (n2 - dof_penalty_2))
        auto_dof = df_num / df_den

    else:
        raise ValueError("Method has to be either 'pooled' or 'unpooled'")

    # Scaling standard error if evaluating half-difference from crossover design
    if is_group_diff:
        se = 0.5 * np.sqrt(var_term)
    else:
        se = np.sqrt(var_term)

    # Option to override dof if needed
    final_dof = dof if dof is not None else auto_dof

    return se, final_dof




#### Analysis by Group Differences (Within-Patient Variability)

"Zoomed-in" analysis that accounts for order/sequence and enables us to discard between-patient variability

In [5]:
def group_diff_analysis(df, method='pooled', dof=None, dof_penalty_1=1, dof_penalty_2=1):
    """
    Function that analyzes data by group treatment differences, eliminating between-patient variability.
    """

    # Defining groups as fresh dfs
    df_A = df[df['Group'] == 'A'].copy()
    df_B = df[df['Group'] == 'B'].copy()

    # Calculating sequence-specific differences 

    # Group A: y1 - y2
    df_A['d'] = df_A['y1'] - df_A['y2']
    # Group B: y2 - y1
    df_B['d'] = df_B['y2'] - df_B['y1']

    # Calculating sample stats
    mean_dA = df_A['d'].mean()
    mean_dB = df_B['d'].mean()
    var_dA = df_A['d'].var(ddof=1)
    var_dB = df_B['d'].var(ddof=1)
    n_A, n_B = len(df_A), len(df_B)

    # Defining estimators 
    est_treatment_effect = 0.5 * (mean_dA + mean_dB)
    est_period_effect = 0.5 * (mean_dA - mean_dB)

    # Calculating standard error and degrees of freedom
    se, df_stat = calculate_standard_error(
        var1 = var_dA,
        var2 = var_dB,
        n1 = n_A,
        n2 = n_B,
        method=method,
        dof=dof,
        dof_penalty_1=dof_penalty_1,
        dof_penalty_2=dof_penalty_2,
        is_group_diff=True
    )

    # Calculating test statistics
    treatment_stat = est_treatment_effect / se
    period_stat = est_period_effect / se

    # Two tailed p-values
    p_val_treatment = 2 * (1 - stats.t.cdf(abs(treatment_stat), df=df_stat))
    p_val_period = 2 * (1 - stats.t.cdf(abs(period_stat), df=df_stat))

    return {
        'treatment_effect': est_treatment_effect,
        'period_effect': est_period_effect,
        'se': se,
        'df': df_stat,
        'treatment_stat': treatment_stat,
        'p_val_treatment': p_val_treatment,
        'period_stat': period_stat,
        'p_val_period': p_val_period
    }




#### Analysis by Overall Difference (Between-Patient Variability)

"Zoomed-out" analysis that does not account for order/sequence and means that we pay a "penalty" in having to include between-patient variability

In [6]:
def overall_diff_analysis(df, method='pooled', dof=None, dof_penalty_1=1, dof_penalty_2=1):
    """
    Function that analyzes data by overall treatment difference, must account for between-patient variability.

    No estimate of period effect, since we need sequence information for that.
    """

    # Reshaping data to group by treatment received (not order)
    df_A_x = df[df['Group'] == 'A'][['y1']].rename(columns={'y1': 'outcome'})
    df_A_y = df[df['Group'] == 'A'][['y2']].rename(columns={'y2': 'outcome'})
    df_B_y = df[df['Group'] == 'B'][['y1']].rename(columns={'y1': 'outcome'})
    df_B_x = df[df['Group'] == 'B'][['y2']].rename(columns={'y2': 'outcome'})   

    # Gathering by Treatment outcomes
    outcomes_X = pd.concat([df_A_x, df_B_x])
    outcomes_Y = pd.concat([df_A_y, df_B_y])

    # Calculating group stats
    mean_X = outcomes_X['outcome'].mean()
    mean_Y = outcomes_Y['outcome'].mean()

    var_X = outcomes_X['outcome'].var(ddof=1)
    var_Y = outcomes_Y['outcome'].var(ddof=1)

    n_X, n_Y = len(outcomes_X), len(outcomes_Y)

    # Defining our estimator
    est_treatment_effect = mean_X - mean_Y

    # Standard error and degrees of freedom
    se, df_stat = calculate_standard_error(
        var1 = var_X,
        var2 = var_Y, 
        n1 = n_X,
        n2 = n_Y,
        method = method,
        dof = dof,
        dof_penalty_1=dof_penalty_1,
        dof_penalty_2=dof_penalty_2,
        is_group_diff=False
    )

    # Calculating test statistic and p-value
    t_stat = est_treatment_effect / se
    p_val_treatment = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_stat))

    return {
        'treatment_effect': est_treatment_effect,
        'se': se,
        'df': df_stat,
        'treatment_stat': t_stat,
        'p_val_treatment': p_val_treatment
    }
    

#### Sanity Check: Single Trial Analysis

Just checking to make sure that all seems to be in working order



In [7]:
print("Single Trial Results")

# Printing true values for reference 
print(f"True Treatment Effect: {true_treatment_effect:.2f} | True Period Effect: {delta:.2f}\n")

# Group Diff, Pooled t-Test

# Calling group diff function 
gd_p = group_diff_analysis(df_trial, method='pooled')

# Printing results
print(f"Group Diff (Pooled t-test):   Treatment Est={gd_p['treatment_effect']:.3f}, Period Est={gd_p['period_effect']:.3f}, SE={gd_p['se']:.3f}, df={gd_p['df']:.1f}, p={gd_p['p_val_treatment']:.4f}")


# Group Diff, Unpooled Normal Deviate
gd_u = group_diff_analysis(df_trial, method='unpooled', dof=np.inf)
print(f"Group Diff (Unpooled Z-test): Treatment Est={gd_u['treatment_effect']:.3f}, Period Est={gd_u['period_effect']:.3f}, SE={gd_u['se']:.3f}, df={gd_u['df']}, p={gd_u['p_val_treatment']:.4f}")


# Overall Diff, Pooled t-Test
od_p = overall_diff_analysis(df_trial, method='pooled')
print(f"Overall Diff (Pooled t-test):   Treatment Est={od_p['treatment_effect']:.3f}, Period Est=N/A,   SE={od_p['se']:.3f}, df={od_p['df']:.1f}, p={od_p['p_val_treatment']:.4f}")


# Overall Diff, Unpooled Normal Deviate
od_u = overall_diff_analysis(df_trial, method='unpooled', dof=np.inf)
print(f"Overall Diff (Unpooled Z-test): Treatment Est={od_u['treatment_effect']:.3f}, Period Est=N/A,   SE={od_u['se']:.3f}, df={od_u['df']}, p={od_u['p_val_treatment']:.4f}")

Single Trial Results
True Treatment Effect: 3.00 | True Period Effect: 1.50

Group Diff (Pooled t-test):   Treatment Est=2.387, Period Est=0.454, SE=0.478, df=27.0, p=0.0000
Group Diff (Unpooled Z-test): Treatment Est=2.387, Period Est=0.454, SE=0.488, df=inf, p=0.0000
Overall Diff (Pooled t-test):   Treatment Est=2.465, Period Est=N/A,   SE=2.869, df=56.0, p=0.3940
Overall Diff (Unpooled Z-test): Treatment Est=2.465, Period Est=N/A,   SE=2.869, df=inf, p=0.3903


#### Running Simulations (MCMC 10,000 Iterations)

In [8]:
# Setting number of simulations to run
n_sims = 10000

# Empty tupperware for metrics
results = {
    'gd_pooled_treatment': [], 'gd_pooled_period': [], 'gd_pooled_se': [],
    'gd_unpooled_treatment': [], 'gd_unpooled_period': [], 'gd_unpooled_se': [],
    'od_pooled_treatment': [], 'od_pooled_se': [],
    'od_unpooled_treatment': [], 'od_unpooled_se': []
}

for _ in range (n_sims):
    sim_df = simulate_crossover_data(n_A, n_B, T_x, T_y, delta, sigma_within, sigma_between, mu_baseline)

    # Simulating group diff analyses
    res_gd_p = group_diff_analysis(sim_df, method='pooled')
    res_gd_u = group_diff_analysis(sim_df, method='unpooled', dof=np.inf)

    # Simulating overall diff analyses
    res_od_p = overall_diff_analysis(sim_df, method='pooled')
    res_od_u = overall_diff_analysis(sim_df, method='unpooled', dof=np.inf)

    # Storing outputs
    results['gd_pooled_treatment'].append(res_gd_p['treatment_effect'])
    results['gd_pooled_period'].append(res_gd_p['period_effect'])
    results['gd_pooled_se'].append(res_gd_p['se'])
    
    results['gd_unpooled_treatment'].append(res_gd_u['treatment_effect'])
    results['gd_unpooled_period'].append(res_gd_u['period_effect'])
    results['gd_unpooled_se'].append(res_gd_u['se'])
    
    results['od_pooled_treatment'].append(res_od_p['treatment_effect'])
    results['od_pooled_se'].append(res_od_p['se'])
    
    results['od_unpooled_treatment'].append(res_od_u['treatment_effect'])
    results['od_unpooled_se'].append(res_od_u['se'])

print("SIMULATION RESULTS")
print(f"True Treatment Effect Target: {true_treatment_effect:.4f}")
print(f"True Period Effect Target:    {delta:.4f}\n")

print("ESTIMATED VALUES")
print(f"E[Treatment Effect] (Group Diff, Pooled):   {np.mean(results['gd_pooled_treatment']):.4f}")
print(f"E[Period Effect] (Group Diff):       {np.mean(results['gd_pooled_period']):.4f}")
print(f"E[Treatment Effect] (Overall Diff, Pooled): {np.mean(results['od_pooled_treatment']):.4f}\n")

print("STANDARD ERRORS")
print(f"Average SE (Group Diff, Pooled):     {np.mean(results['gd_pooled_se']):.4f}")
print(f"Average SE (Group Diff, Unpooled):   {np.mean(results['gd_unpooled_se']):.4f}")
print(f"Average SE (Overall Diff, Pooled):   {np.mean(results['od_pooled_se']):.4f}")
print(f"Average SE (Overall Diff, Unpooled): {np.mean(results['od_unpooled_se']):.4f}")


SIMULATION RESULTS
True Treatment Effect Target: 3.0000
True Period Effect Target:    1.5000

ESTIMATED VALUES
E[Treatment Effect] (Group Diff, Pooled):   3.0086
E[Period Effect] (Group Diff):       1.4877
E[Treatment Effect] (Overall Diff, Pooled): 3.2651

STANDARD ERRORS
Average SE (Group Diff, Pooled):     0.5612
Average SE (Group Diff, Unpooled):   0.5608
Average SE (Overall Diff, Pooled):   2.6706
Average SE (Overall Diff, Unpooled): 2.6706
